<a href="https://colab.research.google.com/github/dani931004/worldbox/blob/master/BGRemover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background Remover Tool
### Step 1: Setup and Dependencies
This cell installs the `rembg` library for background removal and mounts Google Drive.

In [ ]:
!pip install rembg[gpu,pillow] onnxruntime-gpu

from google.colab import drive
import os
from rembg import remove, new_session
from PIL import Image
import glob

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Process Google Drive Folder
Enter your Google Drive folder path below. The script will process `.jpg`, `.jpeg`, and `.png` files.

In [ ]:
import re
import os
import glob
import numpy as np
from google.colab import auth
from googleapiclient.discovery import build
from rembg import remove, new_session
from PIL import Image
import scipy.ndimage as ndimage

# @title Configuration
folder_url = 'https://drive.google.com/drive/folders/1OiYzK5sfM11DerHeFwxwq8KFtWZX9-yS' # @param {type:"string"}

# Using 'isnet-general-use' for high-precision masks
model_name = "isnet-general-use"
session = new_session(model_name)

def resolve_gdrive_url(url):
    match = re.search(r'folders/([a-zA-Z0-9_-]+)', url)
    if not match: return url
    folder_id = match.group(1)
    auth.authenticate_user()
    service = build('drive', 'v3')
    try:
        results = service.files().get(fileId=folder_id, fields="name").execute()
        folder_name = results.get('name')
        path = f"/content/drive/MyDrive/{folder_name}"
        if not os.path.exists(path):
            for root, dirs, _ in os.walk('/content/drive/MyDrive'):
                if folder_name in dirs:
                    return os.path.join(root, folder_name)
        return path
    except Exception as e:
        print(f"Error: {e}")
        return None

def process_background_removal(input_input):
    path = input_input
    if "drive.google.com" in input_input:
        path = resolve_gdrive_url(input_input)
        if not path: return

    if not os.path.exists(path):
        print(f"Path {path} not found.")
        return

    extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(path, ext)))

    for file_path in files:
        if "_bgremoved" in file_path: continue

        file_dir, file_name = os.path.split(file_path)
        name, ext = os.path.splitext(file_name)
        output_path = os.path.join(file_dir, f"{name}_bgremoved.png")

        try:
            print(f"Processing (Ultra Clean & Glossy Mode v2): {file_name}...")
            input_image = Image.open(file_path).convert("RGB")
            original_data = np.array(input_image)

            # Step 1: High Precision Removal
            # bg_threshold=0 is absolute minimum for white backgrounds
            result = remove(
                input_image,
                session=session,
                alpha_matting=True,
                alpha_matting_foreground_threshold=240,
                alpha_matting_background_threshold=0,
                alpha_matting_erode_size=1
            )

            # Step 2: Mask Refinement
            res_array = np.array(result)
            if res_array.shape[2] == 4:
                alpha = res_array[:, :, 3]

                # Create base mask
                binary_mask = alpha > 20

                # Stronger noise removal (opening with 2 iterations to dissolve background fragments)
                cleaned_mask = ndimage.binary_opening(binary_mask, iterations=2)

                # Minimal dilation to keep edges sharp but solid
                expanded_mask = ndimage.binary_dilation(cleaned_mask, iterations=1)

                # Ensure subject is 100% solid
                filled_mask = ndimage.binary_fill_holes(expanded_mask)

                # Composite with original RGB for perfect colors
                final_data = np.zeros((res_array.shape[0], res_array.shape[1], 4), dtype=np.uint8)
                final_data[:, :, :3] = original_data
                final_data[:, :, 3] = np.where(filled_mask, 255, 0).astype(np.uint8)

            output_image = Image.fromarray(final_data)
            output_image.save(output_path)
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    print("Processing complete. Enhanced background cleanup applied.")

process_background_removal(folder_url)

Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 03_28_50 PM.png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 03_51_35 PM.png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 03_58_57 PM (4).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_02_24 PM (1).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_02_24 PM (4).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_02_24 PM (2).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_02_24 PM (3).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_06_17 PM (1).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_06_17 PM (2).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 2026, 04_06_18 PM (3).png...
Processing (Ultra Clean & Glossy Mode v2): ChatGPT Image Jun 11, 202